# Image Enhancer CNN

Small CNN regressor for browser image enhancement. Input is RGB preview `[1, 3, 224, 224]`. Output is `[brightness, contrast, saturation]`.

Training is skipped by default. Run cells as-is to export dummy default weights to `models/enhancer.onnx`.

In [1]:
from pathlib import Path
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from tqdm.auto import tqdm

Path("models").mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [2]:
class EnhancementCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.head = nn.Linear(16, 3)

    def forward(self, image):
        return self.head(self.features(image))

model = EnhancementCNN().to(DEVICE)
model(torch.zeros(1, 3, 224, 224, device=DEVICE))

tensor([[ 0.0771, -0.1498,  0.1127]], grad_fn=<AddmmBackward0>)

In [3]:
class EnhancementDataset(Dataset):
    def __init__(self, samples=None):
        self.samples = samples or []

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image, params = self.samples[index]
        return image.float(), params.float()


def train_model(model, dataset, epochs=5, batch_size=16, lr=1e-3):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.SmoothL1Loss()
    model.train()

    for epoch in tqdm(range(epochs), desc="Training"):
        total = 0.0
        for image, target in loader:
            image = image.to(DEVICE)
            target = target.to(DEVICE)
            
            pred = model(image)
            loss = loss_fn(pred, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total += loss.item() * image.size(0)
        tqdm.write(f"Epoch {epoch + 1} Loss={total / len(dataset):.5f}")

    return model

In [5]:
def export_onnx(model, path):
    model.eval().cpu()
    dummy = torch.zeros(1, 3, 224, 224, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        path,
        input_names=["image"],
        output_names=["params"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    return path

export_onnx(model, Path("models/dummy.onnx"))

C:\Users\Марсель\AppData\Local\Temp\ipykernel_2624\773054438.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


WindowsPath('models/dummy.onnx')